# Bài thực hành: Mạng nơ-ron Tích chập (CNN) và Học chuyển giao (Transfer Learning)

## Phần 1: Cài đặt môi trường và Khởi tạo Google Colab

### Phần 1.1: Làm quen với Google Colab và Thiết lập GPU

Chào mừng bạn đến với bài thực hành Chương 6. Trong bài thực hành này, chúng ta sẽ sử dụng Google Colab, một môi trường Jupyter Notebook trực tuyến hoàn toàn miễn phí, cho phép chạy mã Python trực tiếp trên trình duyệt mà không cần cài đặt cục bộ.

Sổ tay (Notebook) này được cấu tạo từ danh sách các ô (cells). Có hai loại ô chính mà bạn cần nắm rõ:
*   **Ô văn bản (Text cell):** Dùng để hiển thị lý thuyết và hướng dẫn (như ô bạn đang đọc).
*   **Ô mã lệnh (Code cell):** Dùng để viết và thực thi mã Python. Để chạy một ô mã lệnh, bạn nhấp chuột vào ô đó và nhấn tổ hợp phím `Shift + Enter` (hoặc bấm nút ▷ Play ở bên trái ô).

**Kích hoạt phần cứng GPU để tăng tốc huấn luyện:**
Việc huấn luyện các mạng học sâu như Mạng nơ-ron Tích chập (CNN) đòi hỏi khối lượng tính toán khổng lồ. Nếu chỉ sử dụng CPU thông thường, quá trình này có thể mất hàng giờ hoặc nhiều ngày. Việc sử dụng GPU có thể giúp rút ngắn đáng kể thời gian huấn luyện (GPU có thể giúp tăng tốc độ tính toán lên hàng chục lần so với CPU).

Để kích hoạt GPU miễn phí trên Google Colab, bạn hãy thực hiện các bước sau:
1. Trên thanh menu, chọn **Runtime** (Thời gian chạy).
2. Chọn **Change runtime type** (Thay đổi loại thời gian chạy).
3. Trong mục **Hardware accelerator** (Trình tăng tốc phần cứng), mở menu thả xuống và chọn **GPU**.
4. Bấm **Save** (Lưu) để hệ thống cấp phát máy ảo mới có GPU cho bạn.

Sau khi đã lưu thiết lập, hãy chạy ô mã lệnh (Code cell) ngay bên dưới để kiểm tra xem thư viện TensorFlow đã nhận diện được GPU hay chưa.

In [14]:
# Import thư viện học sâu TensorFlow
import tensorflow as tf

# Lấy danh sách các thiết bị GPU vật lý mà TensorFlow nhận diện được trên hệ thống
physical_gpus = tf.config.list_physical_devices("GPU") #

# In ra số lượng GPU đang có
print("Số lượng GPU khả dụng:", len(physical_gpus))

# Kiểm tra điều kiện và thông báo trạng thái
if len(physical_gpus) > 0:
    print("Thông tin chi tiết GPU:", physical_gpus)
    print("Thành công! Máy ảo đã được kết nối GPU. Bạn đã sẵn sàng để huấn luyện mạng CNN.")
else:
    print("Cảnh báo: Chưa kết nối được GPU. Vui lòng làm lại bước thiết lập Runtime ở trên (Runtime -> Change runtime type -> GPU).")

Số lượng GPU khả dụng: 0
Cảnh báo: Chưa kết nối được GPU. Vui lòng làm lại bước thiết lập Runtime ở trên (Runtime -> Change runtime type -> GPU).


### Phần 1.2: Kết nối Google Drive và Khai báo thư viện

**1. Kết nối (Mount) Google Drive**
Lưu ý quan trọng: Google Colab cung cấp máy ảo dùng một lần. Nếu bạn đóng trình duyệt hoặc để Notebook chạy không giám sát quá lâu, máy ảo sẽ bị tắt và toàn bộ dữ liệu bạn tải lên sẽ bị mất sạch.

Để lưu lại dữ liệu quan trọng một cách an toàn (như tập dữ liệu ảnh, hoặc trọng số mô hình sau khi huấn luyện), bạn cần kết nối (mount) Google Drive của mình với Colab. Mặc định, thư mục Drive của bạn sẽ được gắn vào đường dẫn `/content/drive/MyDrive`.

Hãy chạy ô mã lệnh dưới đây và làm theo hướng dẫn cấp quyền:

In [15]:
# Cấp quyền kết nối Google Drive với Google Colab
from google.colab import drive
drive.mount('/content/drive')

# Bạn có thể kiểm tra xem ổ đĩa đã được kết nối chưa bằng lệnh shell liệt kê thư mục:
!ls /content/drive/MyDrive | head -n 5

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
11_training_deep_neural_networks_VN.ipynb
Bài tập Học bổng
Bài tập Học bổng (1)
Bản sao của BÁO CÁO TUẦN 3-4 – THỰC TẬP NHẬN THỨC (09-21 6 2025).gform
Bản sao của ĐĂNG KÝ PHỎNG VẤN TRONG CHƯƠNG TRÌNH “KẾT NỐI DOANH NGHIỆP – PHỎNG VẤN TUYỂN DỤNG THỰC TẬP CHO SINH VIÊN NĂM 2026” NGÀY 11 04 26.gform


**2. Khai báo các thư viện cần thiết**
Để xây dựng và huấn luyện Mạng nơ-ron Tích chập (CNN), chúng ta cần import các công cụ toán học và học sâu cốt lõi.

Trong bước này, chúng ta cũng sẽ cố định "hạt giống ngẫu nhiên" (random seed) bằng lệnh `tf.keras.utils.set_random_seed()`. Lệnh này giúp thiết lập hạt giống ngẫu nhiên cho cả TensorFlow, Python và NumPy, đảm bảo rằng các trọng số ngẫu nhiên khởi tạo trong mạng sẽ giống hệt nhau ở mọi lần chạy, giúp bạn dễ dàng tái tạo và đối chiếu kết quả.

In [16]:
# Import thư viện tính toán mảng và ma trận
import numpy as np

# Import thư viện vẽ biểu đồ và trực quan hóa ảnh
import matplotlib.pyplot as plt

# Import TensorFlow và Keras (API cấp cao để xây dựng mạng học sâu)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Cố định hạt giống ngẫu nhiên để kết quả có thể tái tạo (reproducible)
tf.keras.utils.set_random_seed(42)

# Kiểm tra phiên bản
print("Phiên bản TensorFlow đang sử dụng:", tf.__version__)
print("Đã import các thư viện thành công! Bạn đã hoàn tất Phần 1.")

Phiên bản TensorFlow đang sử dụng: 2.19.0
Đã import các thư viện thành công! Bạn đã hoàn tất Phần 1.


## Phần 2: Xây dựng mạng CNN cơ bản từ con số 0

### Phần 2.1: Xây dựng Khối Trích xuất Đặc trưng (Conv2D & MaxPooling2D)

Trong phần này, chúng ta sẽ bắt đầu xây dựng kiến trúc mạng CNN bằng Keras Sequential API. Khác với mạng nơ-ron truyền thống (ANN) có thể gây bùng nổ tham số khi xử lý ảnh có kích thước lớn, mạng CNN giải quyết bài toán này nhờ cơ chế **kết nối cục bộ** và **chia sẻ trọng số**.

Khối trích xuất đặc trưng của mạng CNN thường bao gồm các lớp sau xếp chồng lên nhau:
*   **Lớp Tích chập (Conv2D):** Sử dụng các bộ lọc (filters) trượt trên ảnh để nhận diện và trích xuất các đặc trưng hình học như đường thẳng, góc cạnh. Ta sẽ sử dụng tham số `padding="same"` để giữ nguyên kích thước không gian của bản đồ đặc trưng đầu ra.
*   **Hàm kích hoạt ReLU:** Thường được gộp trực tiếp vào lớp Conv2D. Hàm ReLU ($max(0, z)$) tính toán cực nhanh và giúp khắc phục điểm yếu chí mạng của mạng nơ-ron là hiện tượng triệt tiêu đạo hàm (vanishing gradients), giúp mô hình hội tụ nhanh hơn.
*   **Lớp Gộp cực đại (MaxPooling2D):** Khác với lớp tích chập, lớp gộp hoàn toàn không có trọng số nào cần học. Nhiệm vụ của nó là trượt một cửa sổ nhỏ (ví dụ 2x2) và chỉ giữ lại giá trị lớn nhất. Quá trình này giúp giảm nhanh kích thước không gian của dữ liệu, tiết kiệm bộ nhớ, giảm tính toán và hạn chế tối đa rủi ro quá khớp (overfitting).

Hãy chạy ô mã lệnh dưới đây để khởi tạo phần đầu tiên của mạng CNN.

In [17]:
# Khởi tạo mô hình tuần tự của Keras
model = keras.Sequential()

# KHỐI 1: Trích xuất đặc trưng cấp thấp
# Lớp Conv2D đầu tiên: Sử dụng 32 bộ lọc (filters), kích thước cửa sổ trượt là 3x3.
# Lưu ý: Bắt buộc phải khai báo input_shape ở lớp đầu tiên.
# Giả sử tập dữ liệu của chúng ta có kích thước 28x28 pixel, 1 kênh màu (ảnh xám).
model.add(layers.Conv2D(filters=32, kernel_size=3, padding="same", activation="relu", input_shape=(28, 28, 1)))

# Lớp Gộp cực đại (MaxPooling2D) với cửa sổ 2x2 sẽ giảm kích thước chiều cao và rộng đi một nửa (còn 14x14)
model.add(layers.MaxPool2D(pool_size=2))

# KHỐI 2: Trích xuất đặc trưng cấp cao hơn
# Theo nguyên tắc thiết kế CNN, khi kích thước không gian của ảnh bị thu nhỏ qua lớp Pooling,
# ta thường tăng số lượng bộ lọc lên (nhân đôi lên 64) để học được nhiều đặc trưng phức tạp hơn.
model.add(layers.Conv2D(filters=64, kernel_size=3, padding="same", activation="relu"))
model.add(layers.MaxPool2D(pool_size=2))

print("Đã khởi tạo thành công Khối Trích xuất Đặc trưng (Conv2D & MaxPooling2D)!")

Đã khởi tạo thành công Khối Trích xuất Đặc trưng (Conv2D & MaxPooling2D)!


### Phần 2.2: Xây dựng Khối Phân loại (Flatten & Dense) và Trực quan hóa mô hình

Sau khi đã trích xuất được các đặc trưng từ ảnh thông qua các lớp Tích chập và Gộp, mạng nơ-ron cần kết hợp những thông tin này lại để gán một nhãn duy nhất cho bức ảnh (trả lời câu hỏi: "Đó là cái gì?"). Để làm được điều này, chúng ta sẽ sử dụng khối phân loại bao gồm các lớp kết nối đầy đủ (Dense / Fully Connected).

Tuy nhiên, do lớp Dense chỉ nhận đầu vào là mảng 1D, trong khi đầu ra của khối Tích chập đang ở dạng mảng 2D/3D, chúng ta cần sử dụng lớp **Làm phẳng (Flatten)**. Lớp này đơn giản là "duỗi thẳng" dữ liệu thành một véc-tơ 1D duy nhất và hoàn toàn không chứa tham số nào cần học.

Lớp đầu ra (Output Layer) nằm ở cuối mạng sẽ chứa số lượng nơ-ron bằng đúng số lượng phân lớp của bài toán (ví dụ: 10 nơ-ron cho tác vụ phân loại 10 loại quần áo hoặc chữ số). Ở lớp này, hàm kích hoạt **Softmax** được sử dụng để tính toán và cung cấp phân phối xác suất cho từng danh mục. Hàm Softmax đảm bảo các xác suất luôn nằm trong khoảng từ 0 đến 1 và tổng của chúng luôn bằng 1.

Hãy chạy ô mã lệnh dưới đây để thêm các lớp này và dùng lệnh `model.summary()` để xem tóm tắt toàn bộ kiến trúc mạng.

In [18]:
# KHỐI 3: Khối phân loại (Classification)
# 1. Duỗi thẳng bản đồ đặc trưng 2D/3D thành véc-tơ 1D
model.add(layers.Flatten())

# 2. Thêm một lớp ẩn kết nối đầy đủ (Dense) với 128 nơ-ron, hàm kích hoạt ReLU
model.add(layers.Dense(units=128, activation="relu"))

# 3. Lớp đầu ra (Output layer) với 10 nơ-ron (cho 10 phân lớp)
# Sử dụng hàm Softmax để xuất ra phân phối xác suất
model.add(layers.Dense(units=10, activation="softmax"))

print("Đã hoàn thiện kiến trúc mạng CNN cơ bản!")

# Trực quan hóa bảng tóm tắt kiến trúc mạng và số lượng tham số
model.summary()

Đã hoàn thiện kiến trúc mạng CNN cơ bản!


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_8 (Conv2D)               │ (None, 28, 28, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_9 (Conv2D)               │ (None, 14, 14, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 3136)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 128)            │       401,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 421,642 (1.61 MB)

 Trainable params: 421,642 (1.61 MB)

 Non-trainable params: 0 (0.00 B)


**Phân tích bảng tóm tắt (Summary):**
Sau khi chạy lệnh `model.summary()`, bạn hãy quan sát bảng kết quả:
*   Cột `Output Shape` cho thấy cách kích thước không gian của ảnh bị thu nhỏ dần qua các lớp Gộp (MaxPooling2D) và cuối cùng bị duỗi thẳng ở lớp Flatten.
*   Cột `Param #` hiển thị số lượng tham số. Bạn sẽ nhận thấy rằng, các lớp kết nối đầy đủ (Dense) sinh ra một số lượng tham số khổng lồ (thường lên tới hàng trăm ngàn) so với các lớp Tích chập.
*   **Cảnh báo:** Mạng nơ-ron có quá nhiều tham số ở các lớp Dense sẽ rất dễ dẫn đến hiện tượng **Quá khớp (Overfitting)**, khiến mô hình "học vẹt" dữ liệu huấn luyện nhưng dự đoán kém trên dữ liệu thực tế mới. Ở Phần 3, chúng ta sẽ học cách khắc phục vấn đề này.

## Phần 3: Áp dụng kỹ thuật Điều chuẩn (Regularization) chống Quá khớp

### Phần 3.1: Nhận diện Quá khớp (Overfitting) và thiết lập Dừng sớm (Early Stopping)

**1. Hiện tượng Quá khớp (Overfitting) là gì?**
Mạng nơ-ron học sâu (như CNN) thường chứa một lượng tham số khổng lồ, mang lại khả năng biểu diễn mạnh mẽ. Tuy nhiên, sự linh hoạt này đi kèm với rủi ro rất lớn: hiện tượng **Quá khớp (Overfitting)**. Quá khớp xảy ra khi một mô hình quá phức tạp so với lượng dữ liệu huấn luyện, khiến nó không chỉ học các quy luật tổng quát mà còn ghi nhớ ("học vẹt") luôn cả những chi tiết nhiễu ngẫu nhiên trong dữ liệu. Hậu quả là mô hình hoạt động cực kỳ tốt trên tập dữ liệu huấn luyện nhưng lại dự đoán rất tệ trên các dữ liệu thực tế mới chưa từng gặp.

**2. Nhận diện Quá khớp qua Đường cong học tập (Learning Curves)**
Cách tốt nhất để nhận biết mô hình có đang bị quá khớp hay không là vẽ và quan sát Đường cong học tập. Bạn hãy chú ý đến hai đường:
*   Đường sai số trên tập huấn luyện (Training Loss).
*   Đường sai số trên tập xác thực (Validation Loss).

Dấu hiệu đặc trưng nhất của một mô hình quá khớp là sự xuất hiện của một **khoảng cách (gap) lớn** giữa hai đường cong này. Cụ thể, sai số huấn luyện sẽ liên tục giảm, nhưng sai số xác thực sau khi giảm đến một mức cực tiểu sẽ bắt đầu chững lại và **tăng ngược trở lại**.

**3. Giải pháp: Kỹ thuật Dừng sớm (Early Stopping)**
Để ngăn chặn tình trạng này, một giải pháp điều chuẩn (regularization) vô cùng đơn giản nhưng hiệu quả là **Dừng sớm (Early Stopping)**.

Thay vì để mô hình huấn luyện hết số chu kỳ (epochs) đã định, cơ chế Dừng sớm sẽ liên tục theo dõi sai số trên tập xác thực và tự động ngắt quá trình huấn luyện ngay khi sai số này chạm đáy. Kỹ thuật này hiệu quả đến mức chuyên gia AI Geoffrey Hinton đã gọi nó là "một bữa trưa miễn phí tuyệt đẹp".

Trong Keras, chúng ta sử dụng `Callback` để thiết lập Early Stopping với hai tham số quan trọng:
*   `patience`: Số lượng chu kỳ được phép chạy tiếp mà không có sự cải thiện nào trên tập xác thực trước khi mô hình dừng hẳn.
*   `restore_best_weights`: Khi được đặt là `True`, mô hình sẽ tự động khôi phục lại bộ trọng số tốt nhất thay vì giữ lại trọng số ở chu kỳ cuối cùng (lúc đã bị quá khớp).

Hãy chạy ô mã lệnh dưới đây để khởi tạo cơ chế Dừng sớm này.

In [19]:
# Import thư viện callback từ keras
from tensorflow.keras.callbacks import EarlyStopping

# Thiết lập cơ chế Dừng sớm (Early Stopping)
early_stopping_cb = EarlyStopping(
    monitor='val_loss',         # Theo dõi sai số trên tập xác thực (Validation Loss)
    patience=10,                # Kiên nhẫn chờ thêm 10 epochs xem sai số có giảm tiếp không
    restore_best_weights=True   # Tự động quay lui (roll back) về bộ trọng số tốt nhất
)

print("Đã thiết lập thành công Callback Early Stopping!")
# Lưu ý: Chúng ta sẽ truyền biến early_stopping_cb này vào hàm model.fit() ở bước huấn luyện.

Đã thiết lập thành công Callback Early Stopping!


### Phần 3.2: Bổ sung Dropout và Tăng cường dữ liệu (Data Augmentation)

Bên cạnh Dừng sớm, chúng ta có thể can thiệp trực tiếp vào dữ liệu và cấu trúc mạng để chống lại hiện tượng Quá khớp bằng hai kỹ thuật cực kỳ mạnh mẽ: Tăng cường dữ liệu và Dropout.

**1. Kỹ thuật Tăng cường dữ liệu (Data Augmentation)**
Mạng CNN cần rất nhiều dữ liệu để tổng quát hóa tốt. Nếu chỉ có lượng ảnh giới hạn, mô hình sẽ ghi nhớ chính xác từng pixel của ảnh gốc. Tăng cường dữ liệu là giải pháp khắc phục sự thiếu hụt này bằng cách tạo ra các biến thể nhân tạo từ tập dữ liệu gốc (ví dụ: lật ngang, xoay nghiêng, phóng to/thu nhỏ). Quá trình này ép mạng phải học các đặc trưng bất biến (invariant features) – tức là dù đối tượng bị xoay hay lật thì mạng vẫn nhận diện được.
*Lưu ý:* Các lớp Tăng cường dữ liệu trong Keras chỉ tự động kích hoạt trong quá trình huấn luyện (Training) và bị tắt khi mô hình thực hiện dự đoán (Inference).

**2. Kỹ thuật Dropout - Sức mạnh của Sự Ngẫu nhiên**
Dropout là kỹ thuật điều chuẩn phổ biến nhất, hoạt động bằng cách ngẫu nhiên "tắt" (loại bỏ) một tỷ lệ $p$ nơ-ron trong mỗi bước huấn luyện. Đối với mạng CNN, tỷ lệ này thường được thiết lập ở mức 40% - 50%. Việc áp dụng Dropout mang lại hai hiệu quả to lớn:
*   **Ngăn chặn sự đồng thích nghi:** Buộc các nơ-ron không được "dựa dẫm" vào nhau mà phải tự trích xuất ra các đặc trưng độc lập hữu ích.
*   **Hiệu ứng Tổ hợp (Ensemble):** Việc thay đổi cấu trúc mạng liên tục ở mỗi bước giống như ta đang huấn luyện và tính trung bình của hàng triệu mạng nơ-ron nhỏ khác nhau, giúp tăng độ chính xác tổng quát hóa.

Hãy chạy ô mã lệnh dưới đây để xây dựng một mạng CNN hoàn chỉnh có nhúng trực tiếp Data Augmentation và Dropout.

In [20]:
# Khởi tạo mô hình tuần tự mới tích hợp các kỹ thuật Điều chuẩn
model_reg = keras.Sequential()

# KHỐI 1: TĂNG CƯỜNG DỮ LIỆU (Data Augmentation)
# Giả sử đầu vào là ảnh màu cỡ 28x28x1
data_augmentation = keras.Sequential([
    layers.RandomFlip(mode="horizontal", seed=42),       # Lật ngang ngẫu nhiên
    layers.RandomRotation(factor=0.05, seed=42),         # Xoay ngẫu nhiên một góc nhỏ
    layers.RandomZoom(height_factor=0.1, seed=42)        # Phóng to/thu nhỏ ngẫu nhiên
], name="data_augmentation")

model_reg.add(data_augmentation)

# KHỐI 2: TRÍCH XUẤT ĐẶC TRƯNG (Conv2D & MaxPool2D)
# Lớp tích chập đầu tiên khai báo input_shape
model_reg.add(layers.Conv2D(filters=32, kernel_size=3, padding="same", activation="relu", input_shape=(28, 28, 1)))
model_reg.add(layers.MaxPool2D(pool_size=2))
model_reg.add(layers.Conv2D(filters=64, kernel_size=3, padding="same", activation="relu"))
model_reg.add(layers.MaxPool2D(pool_size=2))

# KHỐI 3: PHÂN LOẠI TÍCH HỢP DROPOUT
model_reg.add(layers.Flatten())

# Thêm Dropout với tỷ lệ 50% trước lớp Dense ẩn
model_reg.add(layers.Dropout(rate=0.5))
model_reg.add(layers.Dense(units=128, activation="relu"))

# Lớp đầu ra (10 phân lớp)
model_reg.add(layers.Dense(units=10, activation="softmax"))

print("Đã khởi tạo thành công mạng CNN tích hợp Data Augmentation và Dropout!")

# In bảng tóm tắt
# (Lưu ý: Các lớp Dropout và Data Augmentation hoàn toàn không làm tăng số lượng tham số Param # của mạng)
model_reg.summary()

Đã khởi tạo thành công mạng CNN tích hợp Data Augmentation và Dropout!


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ data_augmentation (Sequential)  │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## Phần 4: Học chuyển giao (Transfer Learning) - Chiến lược Trích xuất đặc trưng

### Phần 4.1: Tải mô hình Pre-trained và Đóng băng trọng số (Feature Extraction)

**1. Học chuyển giao (Transfer Learning) là gì?**
Thay vì huấn luyện một mạng nơ-ron sâu khổng lồ từ con số 0, chúng ta có thể tái sử dụng tri thức từ một mạng hiện có đã được huấn luyện trên các tập dữ liệu cực lớn (ví dụ: ImageNet với 1.2 triệu ảnh, 1000 phân lớp) để áp dụng cho một bài toán mới. Việc này mang lại 3 lợi ích khổng lồ: tăng tốc độ hội tụ, tiết kiệm dữ liệu gán nhãn, và cải thiện hiệu suất tổng quát hóa.

**2. Chiến lược Trích xuất đặc trưng (Feature Extraction)**
Đặc thù của mạng CNN là các lớp thấp (nằm gần đầu vào) luôn học được các đặc trưng mang tính tổng quát cao như đoạn thẳng, góc cạnh, hay màu sắc. Dựa vào đặc tính này, chiến lược Trích xuất đặc trưng bao gồm hai bước cốt lõi:
*   **Tải mô hình cơ sở (Base Model):** Tải phần lớn các lớp từ mạng đã huấn luyện, nhưng **bắt buộc phải cắt bỏ lớp phân loại (Fully Connected)** nằm ở đỉnh mạng gốc. Tham số `include_top=False` trong Keras sẽ thực hiện việc này.
*   **Đóng băng trọng số (Freeze):** Thiết lập các lớp của mô hình cơ sở ở trạng thái không thể huấn luyện (`trainable = False`). Điều này đóng vai trò như một "ổ khóa", đảm bảo thuật toán hạ gradient (gradient descent) không làm thay đổi và phá hỏng các trọng số tinh tế đã được tối ưu hóa từ trước.

Keras cung cấp sẵn rất nhiều kiến trúc nổi tiếng như VGG, ResNet, hay Xception. Hãy chạy ô mã lệnh dưới đây để tải mô hình **Xception** làm bộ trích xuất đặc trưng và đóng băng nó.

In [21]:
# Khai báo kích thước ảnh chuẩn đầu vào (Xception và ResNet thường dùng ảnh 224x224 hoặc 299x299)
IMG_SHAPE = (224, 224, 3)

# 1. Tải mô hình cơ sở Xception (Pre-trained Model)
base_model = tf.keras.applications.Xception(
    weights='imagenet',  # Tải bộ trọng số đã được tối ưu hóa sẵn từ tập dữ liệu ImageNet
    include_top=False,   # [RẤT QUAN TRỌNG] Cắt bỏ lớp phân loại ở đỉnh mạng gốc
    input_shape=IMG_SHAPE
)

# 2. Đóng băng toàn bộ trọng số của mô hình cơ sở (Freeze)
# Gradient descent sẽ bỏ qua các lớp này trong quá trình huấn luyện ở Phần 4
base_model.trainable = False

print("Đã tải và đóng băng thành công mô hình cơ sở Xception!")

# Hiển thị bảng tóm tắt
# Quan sát phần cuối bảng, bạn sẽ thấy toàn bộ ~20 triệu tham số của Xception
# đều được chuyển sang trạng thái "Non-trainable params" (Tham số không huấn luyện).
base_model.summary()

Đã tải và đóng băng thành công mô hình cơ sở Xception!


Model: "xception"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1        │ (None, 111, 111,  │        864 │ input_layer_3[0]… │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1_bn     │ (None, 111, 111,  │        128 │ block1_conv1[0][… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1_act    │ (None, 111, 111,  │          0 │ block1_conv1_bn[… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2        │ (None, 109, 109,  │     18,432 │ block1_conv1_act… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2_bn     │ (None, 109, 109,  │        256 │ block1_conv2[0][… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2_act    │ (None, 109, 109,  │          0 │ block1_conv2_bn[… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv1     │ (None, 109, 109,  │      8,768 │ block1_conv2_act… │
│ (SeparableConv2D)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv1_bn  │ (None, 109, 109,  │        512 │ block2_sepconv1[… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv2_act │ (None, 109, 109,  │          0 │ block2_sepconv1_… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv2     │ (None, 109, 109,  │     17,536 │ block2_sepconv2_… │
│ (SeparableConv2D)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv2_bn  │ (None, 109, 109,  │        512 │ block2_sepconv2[… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_12 (Conv2D)  │ (None, 55, 55,    │      8,192 │ block1_conv2_act… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_pool         │ (None, 55, 55,    │          0 │ block2_sepconv2_… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 55, 55,    │        512 │ conv2d_12[0][0]   │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_12 (Add)        │ (None, 55, 55,    │          0 │ block2_pool[0][0… │
│                     │ 128)              │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_sepconv1_act │ (None, 55, 55,    │          0 │ add_12[0][0]    

 Total params: 20,861,480 (79.58 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 20,861,480 (79.58 MB)

### Phần 4.2: Thêm Đầu phân loại tùy chỉnh (Custom Head) và Hoàn thiện mô hình

Sau khi đã tải và đóng băng phần trích xuất đặc trưng (Base Model), bước tiếp theo là thiết kế một cái "Đầu" (Head) mới cho kiến trúc để phù hợp với số lượng phân lớp của bài toán hiện tại.

Thay vì sử dụng lớp Làm phẳng (`Flatten`) như ở mạng CNN cơ bản (có thể sinh ra hàng triệu tham số khi kết nối với mô hình lớn như Xception và gây quá khớp), chúng ta sẽ sử dụng một kỹ thuật hiệu quả hơn rất nhiều:
1. **Lớp Gộp toàn cục (Global Average Pooling 2D):** Lớp này tính giá trị trung bình của toàn bộ mỗi bản đồ đặc trưng. Ý nghĩa của nó là giảm triệt để chiều dữ liệu và số lượng tham số, từ đó chống lại hiện tượng Overfitting cực kỳ hiệu quả.
2. **Lớp Phân loại (Dense):** Lớp đầu ra với số nơ-ron bằng chính xác số lớp của bài toán mới, sử dụng hàm `softmax` để xuất ra xác suất.

Trong phần này, chúng ta sẽ sử dụng **Functional API** của Keras để kết nối các lớp lại với nhau. Hãy chạy ô mã lệnh dưới đây để hoàn thiện kiến trúc:

In [22]:
# Giả sử bài toán mới của chúng ta cần phân loại 10 danh mục (ví dụ: 10 loại động vật)
n_classes = 10

# 1. Áp dụng Lớp Gộp toàn cục (Global Average Pooling 2D)
# Lớp này sẽ nhận đầu ra (output) của mô hình cơ sở Xception làm đầu vào
avg = layers.GlobalAveragePooling2D()(base_model.output)

# 2. Thêm Lớp phân loại (Dense) mới
# Số nơ-ron bằng số lượng phân lớp (n_classes), dùng hàm kích hoạt softmax
class_output = layers.Dense(units=n_classes, activation='softmax')(avg)

# 3. Đóng gói thành một Mô hình hoàn chỉnh (Model)
# Định nghĩa rõ đâu là đầu vào (lấy từ base_model) và đâu là đầu ra (class_output vừa tạo)
model_tl = keras.Model(inputs=base_model.input, outputs=class_output)

print("Đã lắp ráp xong kiến trúc Học chuyển giao (Transfer Learning)!")

# Hiển thị bảng tóm tắt kiến trúc mạng
model_tl.summary()

# Phân tích kết quả (Dành cho sinh viên tự quan sát):
# Khi nhìn vào cuối bảng summary, bạn sẽ thấy:
# - Total params (Tổng số tham số): ~20.8 triệu.
# - Trainable params (Số tham số cần học): Chỉ có khoảng 20,490 tham số (thuộc về lớp Dense mới).
# - Non-trainable params (Số tham số bị đóng băng): ~20.8 triệu (toàn bộ trọng số tinh tế của Xception đã được bảo vệ an toàn).

Đã lắp ráp xong kiến trúc Học chuyển giao (Transfer Learning)!


Model: "functional_16"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1        │ (None, 111, 111,  │        864 │ input_layer_3[0]… │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1_bn     │ (None, 111, 111,  │        128 │ block1_conv1[0][… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1_act    │ (None, 111, 111,  │          0 │ block1_conv1_bn[… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2        │ (None, 109, 109,  │     18,432 │ block1_conv1_act… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2_bn     │ (None, 109, 109,  │        256 │ block1_conv2[0][… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2_act    │ (None, 109, 109,  │          0 │ block1_conv2_bn[… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv1     │ (None, 109, 109,  │      8,768 │ block1_conv2_act… │
│ (SeparableConv2D)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv1_bn  │ (None, 109, 109,  │        512 │ block2_sepconv1[… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv2_act │ (None, 109, 109,  │          0 │ block2_sepconv1_… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv2     │ (None, 109, 109,  │     17,536 │ block2_sepconv2_… │
│ (SeparableConv2D)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv2_bn  │ (None, 109, 109,  │        512 │ block2_sepconv2[… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_12 (Conv2D)  │ (None, 55, 55,    │      8,192 │ block1_conv2_act… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_pool         │ (None, 55, 55,    │          0 │ block2_sepconv2_… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 55, 55,    │        512 │ conv2d_12[0][0]   │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_12 (Add)        │ (None, 55, 55,    │          0 │ block2_pool[0][0… │
│                     │ 128)              │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_sepconv1_act │ (None, 55, 55,    │          0 │ add_12[0][0]    

 Total params: 20,881,970 (79.66 MB)

 Trainable params: 20,490 (80.04 KB)

 Non-trainable params: 20,861,480 (79.58 MB)

## Phần 5: Chiến lược Tinh chỉnh mô hình (Fine-Tuning)

### Phần 5.1: Mở băng (Unfreeze) các lớp trên cùng và Cấu hình Huấn luyện lại

**1. Tinh chỉnh (Fine-Tuning) là gì?**
Sau khi áp dụng chiến lược Trích xuất đặc trưng (Feature Extraction) ở Phần 4 (đóng băng mô hình cơ sở và chỉ huấn luyện lớp phân loại mới trong vài chu kỳ để ổn định trọng số), bước tiếp theo để tối đa hóa hiệu suất là **Tinh chỉnh (Fine-Tuning)**.

Khái niệm này liên quan đến việc mở băng (Unfreeze) một số lớp ẩn ở các tầng trên cùng của mô hình Pre-trained để tiếp tục huấn luyện chúng cùng với lớp Output mới. Mục đích của việc này là giúp các trọng số của các lớp trích xuất đặc trưng cấp cao có thể tự điều chỉnh và thích ứng sát hơn với các đặc thù của tập dữ liệu mới.

**2. Hai quy tắc "sống còn" khi thực hiện Fine-Tuning:**
*   **Tốc độ học (Learning Rate) cực nhỏ:** Bắt buộc phải giảm tốc độ học (nhỏ hơn hoặc bằng 10 lần so với ban đầu) khi tiến hành mở băng. Nếu sử dụng tốc độ học lớn, các bước cập nhật gradient sẽ quá mạnh, làm phá hủy hoàn toàn các trọng số tinh tế đã được tối ưu hóa từ trước.
*   **Số lượng lớp được mở băng:** Quyết định này phụ thuộc vào lượng dữ liệu bạn có. Nếu dữ liệu huấn luyện ít, bạn chỉ nên tinh chỉnh lớp phân loại cuối hoặc 1 vài lớp ẩn trên cùng để ngăn chặn hiện tượng Quá khớp (Overfitting) nghiêm trọng. Chỉ khi có rất nhiều dữ liệu, bạn mới nên cho phép mở băng nhiều lớp ẩn hơn.
*   **Biên dịch lại (Re-compile):** Trong Keras, sau khi thay đổi trạng thái đóng băng/mở băng (`trainable = True/False`) của bất kỳ lớp nào, bạn **bắt buộc** phải biên dịch (`compile`) lại mô hình để các thay đổi có hiệu lực.

Hãy chạy ô mã lệnh dưới đây để tiến hành mở băng khoảng 20 lớp trên cùng của mô hình Xception và thiết lập một tốc độ học thật nhỏ cho quá trình Fine-Tuning.

In [23]:
# 1. Mở băng toàn bộ mô hình cơ sở Xception
base_model.trainable = True

# Xem tổng số lớp hiện có trong mô hình cơ sở
print("Tổng số lớp trong mô hình cơ sở Xception:", len(base_model.layers))

# 2. Quyết định vị trí bắt đầu tinh chỉnh (Fine-tune from)
# Để chống Overfitting vì dữ liệu thường ít, ta sẽ đóng băng lại hầu hết các lớp bên dưới,
# và chỉ để lại khoảng 20 lớp trên cùng ở trạng thái mở băng (trainable = True).
fine_tune_at = len(base_model.layers) - 20

# Lặp qua tất cả các lớp trước vị trí fine_tune_at và đóng băng chúng lại
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

# 3. Biên dịch (Compile) lại mô hình [BẮT BUỘC]
# Khởi tạo bộ tối ưu hóa mới với Tốc độ học (Learning Rate) RẤT NHỎ (ví dụ: 1e-5 thay vì 1e-3)
optimizer_finetune = keras.optimizers.Adam(learning_rate=1e-5)

model_tl.compile(
    optimizer=optimizer_finetune,
    loss='sparse_categorical_crossentropy', # Dùng cho nhãn dạng số nguyên
    metrics=['accuracy']
)

print("Đã mở băng các lớp trên cùng và cấu hình xong cho quá trình Fine-Tuning!")

# Hiển thị lại bảng tóm tắt
# Bạn hãy quan sát: Số lượng "Trainable params" (Tham số cần học) lúc này đã tăng lên đáng kể
# so với ở Phần 4.2 do ta đã mở băng thêm 20 lớp, nhưng phần lớn các tham số ở dưới vẫn là "Non-trainable".
model_tl.summary()

Tổng số lớp trong mô hình cơ sở Xception: 132
Đã mở băng các lớp trên cùng và cấu hình xong cho quá trình Fine-Tuning!


Model: "functional_16"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1        │ (None, 111, 111,  │        864 │ input_layer_3[0]… │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1_bn     │ (None, 111, 111,  │        128 │ block1_conv1[0][… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1_act    │ (None, 111, 111,  │          0 │ block1_conv1_bn[… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2        │ (None, 109, 109,  │     18,432 │ block1_conv1_act… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2_bn     │ (None, 109, 109,  │        256 │ block1_conv2[0][… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2_act    │ (None, 109, 109,  │          0 │ block1_conv2_bn[… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv1     │ (None, 109, 109,  │      8,768 │ block1_conv2_act… │
│ (SeparableConv2D)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv1_bn  │ (None, 109, 109,  │        512 │ block2_sepconv1[… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv2_act │ (None, 109, 109,  │          0 │ block2_sepconv1_… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv2     │ (None, 109, 109,  │     17,536 │ block2_sepconv2_… │
│ (SeparableConv2D)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv2_bn  │ (None, 109, 109,  │        512 │ block2_sepconv2[… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_12 (Conv2D)  │ (None, 55, 55,    │      8,192 │ block1_conv2_act… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_pool         │ (None, 55, 55,    │          0 │ block2_sepconv2_… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 55, 55,    │        512 │ conv2d_12[0][0]   │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_12 (Add)        │ (None, 55, 55,    │          0 │ block2_pool[0][0… │
│                     │ 128)              │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_sepconv1_act │ (None, 55, 55,    │          0 │ add_12[0][0]    

 Total params: 20,881,970 (79.66 MB)

 Trainable params: 7,346,866 (28.03 MB)

 Non-trainable params: 13,535,104 (51.63 MB)

### Phần 5.2: Tiến hành Huấn luyện tinh chỉnh (Fine-Tuning) và Đánh giá

Sau khi đã mở băng (unfreeze) các lớp trên cùng và biên dịch lại mô hình, bước cuối cùng là tiến hành quá trình huấn luyện thực sự.

**Lưu ý quan trọng nhất của Fine-Tuning:**
Như đã đề cập ở bước trước, bạn **bắt buộc phải sử dụng tốc độ học (learning rate) rất nhỏ** (nhỏ hơn ít nhất 10 lần so với ban đầu). Nếu sử dụng tốc độ học lớn, các bước cập nhật gradient sẽ diễn ra quá mạnh, làm phá hủy hoàn toàn các trọng số tinh tế đã được tối ưu hóa từ trước.

Việc huấn luyện lại với tốc độ học nhỏ này sẽ giúp các đặc trưng cấp cao tự điều chỉnh sát hơn với đặc thù của tập dữ liệu mới, trong khi vẫn giữ được nền tảng nhận diện hình ảnh xuất sắc của mô hình cơ sở.

Hãy chạy ô mã lệnh dưới đây để bắt đầu huấn luyện. Giống như ở Phần 3, chúng ta tiếp tục sử dụng cơ chế Dừng sớm (Early Stopping) để ngăn chặn hiện tượng Quá khớp (Overfitting) trong quá trình Fine-Tuning.

In [24]:
# Giả định rằng bạn đã có tập dữ liệu (train_dataset, validation_dataset) đã được tiền xử lý.
# (Bạn sẽ tự xây dựng pipeline dữ liệu này trong phần Bài tập).

print("Bắt đầu quá trình Fine-Tuning...")

# Tiếp tục huấn luyện mô hình (Fine-tuning)
# Ta gọi lại hàm fit() với bộ tối ưu hóa mới (đã set learning_rate rất nhỏ ở bước trước)
'''
# Đoạn code huấn luyện thực tế (bỏ comment khi chạy với dữ liệu thật):
history_finetune = model_tl.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=50,                      # Có thể đặt số epochs lớn vì đã có Early Stopping
    callbacks=[early_stopping_cb]   # Sử dụng lại callback Dừng sớm đã định nghĩa ở Phần 3
)

# Sau khi huấn luyện xong, bạn có thể đánh giá mô hình trên tập Test
test_loss, test_acc = model_tl.evaluate(test_dataset)
print(f"Độ chính xác trên tập Test sau khi Fine-Tuning: {test_acc * 100:.2f}%")
'''

print("Mô hình đã sẵn sàng để huấn luyện với dữ liệu của bạn!")
print("Hoàn tất Phần 5. Bạn đã nắm vững hai chiến lược quan trọng nhất của Transfer Learning: Trích xuất đặc trưng và Tinh chỉnh.")

Bắt đầu quá trình Fine-Tuning...
Mô hình đã sẵn sàng để huấn luyện với dữ liệu của bạn!
Hoàn tất Phần 5. Bạn đã nắm vững hai chiến lược quan trọng nhất của Transfer Learning: Trích xuất đặc trưng và Tinh chỉnh.


## Phần 6: Mở rộng CNN - Kiến trúc Đa tác vụ (Multi-task Learning) cho Định vị

### Phần 6.1: Xây dựng nhánh Định vị đối tượng (Localization)

**1. Từ Nhận diện đến Định vị không gian**
Bài toán phân loại hình ảnh (Image Classification) nhằm trả lời câu hỏi "Trong ảnh có đối tượng gì?", nhưng nhiều ứng dụng thực tế đòi hỏi chúng ta phải trả lời thêm câu hỏi "Nó nằm ở đâu?". Định vị đối tượng (Localization) giải quyết vấn đề này bằng cách vẽ một hộp giới hạn (Bounding box) bao quanh đối tượng.

Khác với phân loại, định vị là một bài toán **Hồi quy (Regression)** chuyên dự đoán các giá trị liên tục. Một cách tiếp cận phổ biến là dự đoán 4 giá trị số để định hình hộp giới hạn: tọa độ ngang và dọc của tâm đối tượng (x, y), cùng với chiều cao (h) và chiều rộng (w).

**2. Mô hình Đa tác vụ (Multi-task Learning)**
Thay vì xây dựng hai mô hình riêng biệt (một cho phân loại, một cho định vị), chúng ta có thể áp dụng kiến trúc Học đa tác vụ. Bằng cách rẽ nhánh từ một mô hình cơ sở (Pretrained Model) duy nhất, mạng nơ-ron có thể chia sẻ các đặc trưng đã học để giải quyết đồng thời cả hai bài toán, giúp mô hình học nhanh hơn và tiết kiệm tài nguyên tính toán.

Kiến trúc rẽ nhánh bao gồm:
*   **Nhánh 1 (Phân loại):** Lớp kết nối đầy đủ (Dense) có số nơ-ron bằng số lượng lớp, dùng hàm kích hoạt Softmax và hàm mất mát Cross-entropy.
*   **Nhánh 2 (Định vị):** Bổ sung song song một lớp Dense thứ hai chỉ gồm 4 nơ-ron để dự đoán 4 tọa độ (x, y, w, h). Nhánh này sẽ được huấn luyện bằng hàm mất mát MSE (Sai số toàn phương trung bình).

Hãy chạy ô mã lệnh dưới đây để lắp ráp mô hình đa tác vụ này bằng Keras Functional API.

In [25]:
# Giả sử chúng ta tiếp tục sử dụng mô hình Xception làm bộ trích xuất đặc trưng (base_model) đã tạo ở Phần 4
# 1. Trích xuất đặc trưng chung (Shared Features)
avg = layers.GlobalAveragePooling2D()(base_model.output)

# 2. Xây dựng NHÁNH 1: Phân loại (Classification)
# Lớp Dense phân loại 10 lớp, hàm kích hoạt Softmax
class_output = layers.Dense(units=10, activation='softmax', name='class_output')(avg)

# 3. Xây dựng NHÁNH 2: Định vị (Localization)
# Bổ sung song song lớp Dense thứ hai, có đúng 4 nơ-ron để dự đoán (x, y, w, h)
# Không dùng hàm kích hoạt (hoặc hàm tuyến tính) vì đây là bài toán hồi quy giá trị liên tục
loc_output = layers.Dense(units=4, name='loc_output')(avg)

# 4. Đóng gói thành Mô hình Đa tác vụ (Multi-task Model)
# Khai báo model nhận 1 đầu vào nhưng trả về 2 đầu ra
model_multitask = keras.Model(inputs=base_model.input,
                              outputs=[class_output, loc_output])

# 5. Biên dịch mô hình với nhiều hàm mất mát (Multi-loss)
optimizer = keras.optimizers.Adam(learning_rate=1e-4)

# Gán hàm mất mát riêng cho từng đầu ra và thiết lập trọng số (loss_weights)
# Trọng số giúp mô hình cân bằng việc học giữa 2 tác vụ
model_multitask.compile(
    optimizer=optimizer,
    loss={
        'class_output': 'sparse_categorical_crossentropy',
        'loc_output': 'mse'  # Sử dụng MSE (Sai số toàn phương trung bình) cho định vị
    },
    loss_weights={
        'class_output': 0.8,
        'loc_output': 0.2    # Tuỳ chỉnh trọng số tùy thuộc vào tác vụ nào bạn ưu tiên hơn
    },
    metrics={'class_output': ['accuracy']}
)

print("Đã xây dựng thành công kiến trúc CNN Đa tác vụ (Phân loại + Định vị)!")

# Bạn có thể chạy model_multitask.summary() để xem cách mạng rẽ thành 2 nhánh ở phần cuối.

Đã xây dựng thành công kiến trúc CNN Đa tác vụ (Phân loại + Định vị)!


### Phần 6.2: Hàm mất mát và Chỉ số đánh giá không gian (IoU)

**1. Hàm mất mát cho nhánh Định vị (MSE)**
Như đã thiết lập ở Phần 6.1, vì việc dự đoán tọa độ hộp giới hạn (x, y, w, h) là một bài toán Hồi quy (Regression), chúng ta sử dụng hàm mất mát Sai số toàn phương trung bình (Mean Squared Error - MSE) để huấn luyện nhánh này.

Tuy nhiên, MSE chỉ hoạt động tốt trong việc tối ưu hóa (cập nhật trọng số) chứ không phải là một thước đo tốt để đánh giá hiệu suất định vị thực tế. Lý do là MSE không phản ánh đúng tỷ lệ bao phủ thực tế giữa hộp dự đoán và hộp chuẩn (Ground truth). Ví dụ: một sai số 10 pixel ở một hộp giới hạn kích thước nhỏ sẽ nghiêm trọng hơn rất nhiều so với sai số 10 pixel ở một hộp kích thước khổng lồ, nhưng hàm MSE lại phạt hai trường hợp này nặng như nhau.

**2. Thước đo Độ chính xác Không gian: Chỉ số IoU (Intersection over Union)**
Để khắc phục nhược điểm của MSE khi đánh giá, thước đo phổ biến nhất được sử dụng trong bài toán phát hiện đối tượng là **IoU** (còn được gọi là chỉ số Jaccard).

*   **Công thức:** IoU = (Diện tích phần Giao nhau) $\div$ (Diện tích phần Hợp nhất) của hộp dự đoán và hộp chuẩn.
*   **Biên độ:** Giá trị của IoU dao động từ 0 (hai hộp hoàn toàn không chạm nhau/không có phần giao) đến 1 (hai hộp khớp nhau hoàn hảo tuyệt đối).
*   **Ngưỡng đánh giá:** Trong các bài toán thực tế, một dự đoán định vị thường được coi là "Đúng" (True Positive) nếu nhãn phân loại chính xác VÀ chỉ số IoU lớn hơn một ngưỡng quy định (thông thường là IoU > 0.5).

Hãy chạy ô mã lệnh dưới đây để tự xây dựng một hàm tính toán IoU bằng Python. Việc tự viết code tính toán phần giao và phần hợp sẽ giúp bạn hiểu sâu sắc bản chất toán học của chỉ số đánh giá quan trọng này.

In [27]:
import numpy as np

def calculate_iou(boxA, boxB):
    """
    Hàm tính toán chỉ số Intersection over Union (IoU) giữa 2 hộp giới hạn.
    Giả sử định dạng đầu vào của hộp là: [x_center, y_center, width, height]
    """
    # BƯỚC 1: Chuyển đổi từ tọa độ tâm sang tọa độ các góc [x_min, y_min, x_max, y_max]
    boxA_x_min, boxA_y_min = boxA[0] - boxA[2]/2, boxA[1] - boxA[3]/2
    boxA_x_max, boxA_y_max = boxA[0] + boxA[2]/2, boxA[1] + boxA[3]/2

    boxB_x_min, boxB_y_min = boxB[0] - boxB[2]/2, boxB[1] - boxB[3]/2
    boxB_x_max, boxB_y_max = boxB[0] + boxB[2]/2, boxB[1] + boxB[3]/2

    # BƯỚC 2: Tìm tọa độ của vùng Giao nhau (Intersection)
    inter_x_min = max(boxA_x_min, boxB_x_min)
    inter_y_min = max(boxA_y_min, boxB_y_min)
    inter_x_max = min(boxA_x_max, boxB_x_max)
    inter_y_max = min(boxA_y_max, boxB_y_max)

    # Tính diện tích phần Giao nhau (nếu không giao nhau, kích thước sẽ gán bằng 0)
    inter_width = max(0, inter_x_max - inter_x_min)
    inter_height = max(0, inter_y_max - inter_y_min)
    intersection_area = inter_width * inter_height

    # BƯỚC 3: Tính diện tích phần Hợp nhất (Union)
    # Diện tích Hợp = Diện tích A + Diện tích B - Diện tích Giao
    boxA_area = boxA[2] * boxA[3]
    boxB_area = boxB[2] * boxB[3]
    union_area = boxA_area + boxB_area - intersection_area

    # BƯỚC 4: Tính chỉ số IoU
    if union_area == 0:
        return 0
    iou = intersection_area / union_area
    return iou

# --- KIỂM TRA CHỨC NĂNG CỦA HÀM ---

# 1. Ground Truth (Hộp thực tế): Tâm ở (50, 50), rộng 40, cao 40
ground_truth_box = np.array([50, 50, 40, 40])

# 2. Prediction 1 (Dự đoán tốt): Lệch tâm một chút, kích thước gần giống
pred_box_good = np.array([52, 52, 38, 42])

# 3. Prediction 2 (Dự đoán kém): Lệch tâm hoàn toàn, kích thước nhỏ
pred_box_bad = np.array([150, 150, 20, 20])

# Tính toán
iou_good = calculate_iou(ground_truth_box, pred_box_good)
iou_bad = calculate_iou(ground_truth_box, pred_box_bad)

print("--- KẾT QUẢ ĐÁNH GIÁ ĐỊNH VỊ (IoU) ---")
print(f"Chỉ số IoU cho Dự đoán TỐT: {iou_good:.2f} (Khớp phần lớn, > 0.5)")
print(f"Chỉ số IoU cho Dự đoán KÉM: {iou_bad:.2f} (Không giao nhau, = 0.0)")

print("\n🎉 Chúc mừng! Bạn đã hoàn thành toàn bộ nội dung lý thuyết và thực hành thiết kế của Chương 6.")

--- KẾT QUẢ ĐÁNH GIÁ ĐỊNH VỊ (IoU) ---
Chỉ số IoU cho Dự đoán TỐT: 0.82 (Khớp phần lớn, > 0.5)
Chỉ số IoU cho Dự đoán KÉM: 0.00 (Không giao nhau, = 0.0)

🎉 Chúc mừng! Bạn đã hoàn thành toàn bộ nội dung lý thuyết và thực hành thiết kế của Chương 6.


## Phần 7: Bài tập thực hành (Assignments)
### Bài tập 1: Ứng dụng Pretrained Model tức thời

**Yêu cầu:**
Sử dụng một mô hình đã được huấn luyện sẵn (Pretrained Model) để nhận diện một bức ảnh bất kỳ mà bạn tải xuống từ Internet.

**Hướng dẫn sơ bộ:**
1. **Tải mô hình:** Keras cho phép tải các mô hình tiêu chuẩn như ResNet-50 cực kỳ dễ dàng thông qua gói `tf.keras.applications`. Hãy khởi tạo mô hình với tham số `weights="imagenet"` để tải bộ trọng số đã được tối ưu hóa. Lần này ta dùng mô hình để dự đoán trực tiếp nên giữ nguyên lớp phân loại ở đỉnh (không dùng `include_top=False`).
2. **Tiền xử lý ảnh:** Mô hình ResNet-50 yêu cầu đầu vào là ảnh màu có kích thước 224 x 224 pixel. Bạn cần tải ảnh, thay đổi kích thước (resize), và chuyển đổi thành mảng NumPy.
3. **Chuẩn hóa:** Gọi hàm `preprocess_input` của resnet50 để chuẩn hóa giá trị pixel theo đúng cách mô hình đã được huấn luyện.
4. **Dự đoán:** Sử dụng `model.predict()`. Đầu ra sẽ là một ma trận chứa 1000 xác suất (cho 1000 lớp của ImageNet). Hãy dùng hàm `decode_predictions(..., top=3)` để in ra 3 nhãn có xác suất cao nhất bằng chữ cho dễ đọc.

In [ ]:
# --- KHUNG CODE BÀI TẬP 1 ---
import urllib.request
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input, decode_predictions
import numpy as np

# 1. Tải mô hình ResNet50 (đã huấn luyện trên ImageNet)
# TODO: Viết code khởi tạo model tại đây
model_bt1 = ResNet50(weights="imagenet") # Mẫu

# 2. Tải một bức ảnh từ Internet (Thay URL bằng ảnh bạn thích)
image_url = "https://upload.wikimedia.org/wikipedia/commons/3/3a/Cat03.jpg"
image_path = "my_test_image.jpg"
urllib.request.urlretrieve(image_url, image_path)

# 3. Đọc và tiền xử lý ảnh
# TODO: Dùng image.load_img() để đọc ảnh và ép kích thước về target_size=(224, 224)
# TODO: Chuyển ảnh thành mảng numpy (image.img_to_array)
# TODO: Thêm chiều batch size vào mảng (np.expand_dims)
# TODO: Chuẩn hóa mảng bằng preprocess_input()

# 4. Dự đoán và in kết quả
# TODO: Gọi model_bt1.predict()
# TODO: Dùng decode_predictions() in ra top 3 nhãn

### Bài tập 2: Xây dựng và Tối ưu CNN từ con số 0 (Từ A-Z)

**Yêu cầu:**
Tự thiết kế một mạng CNN hoàn chỉnh để phân loại tập dữ liệu Fashion MNIST (Gồm 70.000 ảnh kích thước 28x28 pixel chia thành 10 loại trang phục). Cố gắng đạt độ chính xác trên tập Test lớn hơn 90%.

**Hướng dẫn sơ bộ:**
1. **Chuẩn bị dữ liệu:** Tập Fashion MNIST đã được tích hợp sẵn trong Keras. Chú ý rằng ảnh ban đầu có giá trị pixel từ 0-255, bạn cần chia cho 255.0 để đưa về khoảng 0-1 nhằm giúp mô hình hội tụ tốt hơn.
2. **Thiết kế kiến trúc:**
    * Hãy bắt đầu với một lớp `Conv2D` có 32 hoặc 64 bộ lọc (kích thước 3x3), nhớ khai báo `input_shape=(28, 28, 1)`.
    * Xen kẽ các lớp `Conv2D` (với hàm kích hoạt ReLU) và `MaxPooling2D` (kích thước 2x2) để giảm chiều không gian và tăng số lượng đặc trưng.
    * Kết thúc phần trích xuất bằng lớp `Flatten()`.
    * Thêm phần phân loại với 1-2 lớp `Dense` ẩn. **Lưu ý:** Lớp Dense sinh ra rất nhiều tham số, hãy nhớ thêm `Dropout` (tỷ lệ khoảng 0.5) để chống Overfitting.
    * Lớp Output cuối cùng phải có 10 nơ-ron và hàm kích hoạt `softmax`.
3. **Huấn luyện:** Sử dụng `sparse_categorical_crossentropy` làm hàm mất mát (vì nhãn của chúng ta là số nguyên từ 0-9). Thiết lập `EarlyStopping` để tìm số Epoch tối ưu.

In [ ]:
# --- KHUNG CODE BÀI TẬP 2 ---
# Tải tập dữ liệu Fashion MNIST
(X_train_full, y_train_full), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()

# 1. Tiền xử lý dữ liệu:
# TODO: Chuẩn hóa pixel về và reshape thêm kênh màu (nếu cần)

# 2. Xây dựng mô hình CNN
model_bt2 = keras.Sequential([
    # TODO: Thêm các lớp Conv2D, MaxPooling2D, Dropout, Flatten, Dense...
])

# 3. Biên dịch và cấu hình Early Stopping
# TODO: Gọi model.compile(...)
# TODO: Định nghĩa early_stopping_cb

# 4. Huấn luyện và Đánh giá
# TODO: Gọi model.fit(...)
# TODO: Gọi model.evaluate(...) trên tập X_test, y_test

### Bài tập 3: Quy trình Transfer Learning và Fine-tuning chuẩn

**Yêu cầu:**
Áp dụng kỹ thuật Học chuyển giao (Transfer Learning) để phân loại một tập dữ liệu ảnh cỡ nhỏ (Ví dụ: tập dữ liệu chó/mèo hoặc tập `tf_flowers` gồm 5 loại hoa).

**Hướng dẫn sơ bộ:**
Bài tập này yêu cầu bạn thực hiện đầy đủ 2 giai đoạn cốt lõi của Transfer Learning đã học ở Phần 4 và Phần 5:
1. **Giai đoạn 1 (Trích xuất đặc trưng):**
    * Tải một mô hình cơ sở (như Xception, MobileNetV2, hoặc VGG16) với `include_top=False`.
    * **Đóng băng (Freeze)** toàn bộ trọng số của mô hình cơ sở này (`trainable = False`).
    * Gắn thêm một cái "Đầu" (Head) mới gồm lớp `GlobalAveragePooling2D` và lớp `Dense` phân loại tương ứng với số lớp của tập dữ liệu mới.
    * Huấn luyện mô hình trong khoảng 3-5 epochs để lớp Dense mới học được các biểu diễn cơ bản.
2. **Giai đoạn 2 (Tinh chỉnh - Fine-Tuning):**
    * **Mở băng (Unfreeze)** một số lớp ở tầng trên cùng của mô hình cơ sở (hoặc mở băng toàn bộ nếu dữ liệu tương đối nhiều).
    * **CẢNH BÁO QUAN TRỌNG:** Biên dịch lại mô hình và bắt buộc phải thiết lập tốc độ học (Learning Rate) thật nhỏ (ví dụ: `learning_rate=1e-5`) để không phá hủy các trọng số đã được tối ưu hóa từ trước.
    * Tiếp tục huấn luyện thêm nhiều epochs (nhớ dùng Early Stopping).

In [ ]:
# --- KHUNG CODE BÀI TẬP 3 ---
import tensorflow_datasets as tfds

# Tải tập dữ liệu Hoa (tf_flowers)
# Có 5 lớp hoa: dandelion, daisy, tulips, sunflowers, roses
dataset, info = tfds.load("tf_flowers", as_supervised=True, with_info=True)
n_classes = info.features["label"].num_classes

# (Phần tách tập Train/Val/Test và tiền xử lý ảnh sinh viên tự thực hiện hoặc tham khảo code mẫu của Keras)
# Giả sử bạn đã có train_set và valid_set...

# GIAI ĐOẠN 1: FEATURE EXTRACTION
# TODO: Tải base_model (Ví dụ: Xception) với include_top=False
# TODO: Đóng băng base_model
# TODO: Xây dựng model mới bằng cách thêm GlobalAveragePooling2D và Dense(n_classes, activation="softmax")
# TODO: Compile và fit trong khoảng 3-5 epochs

# GIAI ĐOẠN 2: FINE-TUNING
# TODO: Unfreeze base_model (hoặc một phần của nó)
# TODO: Compile lại model với learning_rate CỰC NHỎ (VD: 1e-5)
# TODO: Tiếp tục fit model với Early Stopping